# UW Tooling: Reason Codes & Underwriter Cards
## Comparing 3 Driver Display Options

This notebook shows:
1. Component breakdown (40/25/25/10 weighted)
2. Variable-level drivers with their weights
3. **Three different approaches to reason codes**
4. Underwriter card template

## Cell 1: Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict

csv_path = r'C:\Box\Box\BOX Subhashree Singh\Business\PAS\physician_scores_20260429_141959.csv'

print("Loading data...")
df = pd.read_csv(csv_path, low_memory=False)
print(f"✅ Loaded {len(df):,} physicians\n")

# Define variable weights from scoring_logic_summary.txt
COMPONENT_WEIGHTS = {
    'adequacy': 0.40,
    'capacity': 0.25,
    'appetite': 0.25,
    'environment': 0.10
}

VARIABLE_WEIGHTS = {
    'adequacy': {
        'score_total_loss_cost': 0.20,
        'score_indemnity_loss_cost': 0.10,
        'score_expense_loss_cost': 0.05,
        'score_total_frequency': 0.10,
        'score_indemnity_frequency': 0.05,
        'score_total_severity': 0.05,
        'score_actual_loss_ratio': 0.10,
        'score_loss_free_years': 0.10,
        'score_limits_to_premium': 0.10,
    },
    'capacity': {
        'score_risk_count': 0.10,
        'capacity_score': 0.30,  # placeholder for missing per/aggregate limits
    },
    'appetite': {
        'score_specialty_risk_tier': 0.25,
        'score_state_venue_risk': 0.15,
        'score_years_since_graduation': 0.10,
        'score_rvu_ratio': 0.10,
        'score_tenure_with_carrier': 0.05,
        'score_practice_size': 0.05,
    },
    'environment': {
        'score_income_inequality': 0.05,
        'score_population_density': 0.05,
        'score_violent_crime_rate': 0.03,
        'score_pct_uninsured': 0.02,
    }
}

print("Component weights (applied at composite level):")
for comp, weight in COMPONENT_WEIGHTS.items():
    print(f"  {comp.capitalize()}: {weight:.0%}")

print(f"\nTotal variables tracked: {sum(len(v) for v in VARIABLE_WEIGHTS.values())}")

Loading data...
✅ Loaded 165,623 physicians

Component weights (applied at composite level):
  Adequacy: 40%
  Capacity: 25%
  Appetite: 25%
  Environment: 10%

Total variables tracked: 21


## Cell 2: Helper Functions

In [2]:
def get_risk_category(score):
    """Convert composite score to risk category."""
    if pd.isna(score):
        return "Unknown"
    if score <= 3:
        return "Low Risk"
    elif score <= 5:
        return "Medium Risk"
    else:
        return "High Risk"

def get_variable_label(col_name):
    """Convert column name to readable label."""
    # Remove 'score_' prefix and capitalize
    label = col_name.replace('score_', '').replace('_', ' ').title()
    return label

def option_a_top_3_drivers(row):
    """Option A: Top 3 sub-score drivers (highest sub-scores)."""
    all_scores = {}
    
    for component, variables in VARIABLE_WEIGHTS.items():
        for col, weight in variables.items():
            if col in df.columns:
                val = row[col]
                if pd.notna(val) and val > 0:
                    all_scores[col] = val
    
    # Sort by score descending, get top 3
    top_3 = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)[:3]
    
    reasons = []
    for col, score in top_3:
        label = get_variable_label(col)
        # Find which component this belongs to
        component = next((c for c, v in VARIABLE_WEIGHTS.items() if col in v), 'Unknown')
        reasons.append(f"{label} ({component.capitalize()} {score:.1f}/10)")
    
    return reasons

def option_b_high_scores(row):
    """Option B: All sub-scores >= 7 (strong outliers above average)."""
    outliers = {}
    
    for component, variables in VARIABLE_WEIGHTS.items():
        for col, weight in variables.items():
            if col in df.columns:
                val = row[col]
                if pd.notna(val) and val >= 7:
                    outliers[col] = val
    
    # Sort by score descending
    sorted_outliers = sorted(outliers.items(), key=lambda x: x[1], reverse=True)
    
    reasons = []
    for col, score in sorted_outliers:
        label = get_variable_label(col)
        component = next((c for c, v in VARIABLE_WEIGHTS.items() if col in v), 'Unknown')
        reasons.append(f"{label} ({component.capitalize()} {score:.1f}/10)")
    
    return reasons if reasons else ["No significant risk factors"]

def option_c_deviation_from_5(row):
    """Option C: Sub-scores that deviate most from 5 (biggest deviations)."""
    deviations = {}
    
    for component, variables in VARIABLE_WEIGHTS.items():
        for col, weight in variables.items():
            if col in df.columns:
                val = row[col]
                if pd.notna(val) and val > 0:
                    dev = abs(val - 5)  # deviation from portfolio average
                    if dev > 0:
                        deviations[col] = (val, dev)
    
    # Sort by absolute deviation descending, get top 3
    top_3 = sorted(deviations.items(), key=lambda x: x[1][1], reverse=True)[:3]
    
    reasons = []
    for col, (score, dev) in top_3:
        label = get_variable_label(col)
        component = next((c for c, v in VARIABLE_WEIGHTS.items() if col in v), 'Unknown')
        direction = "High" if score > 5 else "Low"
        reasons.append(f"{label}: {direction} ({component.capitalize()} {score:.1f}/10)")
    
    return reasons

print("✅ Helper functions loaded")

✅ Helper functions loaded


## Cell 3: Build Sample Underwriter Cards (First 5 Physicians)

In [3]:
# Sample 5 physicians with different risk profiles
sample_indices = [
    df[df['composite_score'] == 2].index[0] if len(df[df['composite_score'] == 2]) > 0 else 0,  # Low risk
    df[df['composite_score'] == 4].index[100] if len(df[df['composite_score'] == 4]) > 100 else 1,  # Medium
    df[df['composite_score'] == 5].index[100] if len(df[df['composite_score'] == 5]) > 100 else 2,  # Medium
    df[df['composite_score'] == 7].index[0] if len(df[df['composite_score'] == 7]) > 0 else 3,  # High risk
    df[df['composite_score'] == 8].index[0] if len(df[df['composite_score'] == 8]) > 0 else 4,  # Highest risk
]

for idx_num, idx in enumerate(sample_indices, 1):
    physician = df.iloc[idx]
    npi = physician['NPI']
    specialty = physician['OL_RISK_SPECIALTY_DESC']
    state = physician['ST']
    composite = physician['composite_score']
    
    print(f"\n{'='*100}")
    print(f"SAMPLE {idx_num}: {specialty} in {state} | NPI: {npi}")
    print(f"{'='*100}")
    print(f"\nComposite Score: {composite:.0f}/10 | {get_risk_category(composite)}")
    print(f"  Adequacy:    {physician['adequacy_score']:.2f}/10 (40% weight)")
    print(f"  Capacity:    {physician['capacity_score']:.2f}/10 (25% weight)")
    print(f"  Appetite:    {physician['appetite_score']:.2f}/10 (25% weight)")
    print(f"  Environment: {physician['environment_score']:.2f}/10 (10% weight)")
    
    # Generate all 3 reason code options
    option_a = option_a_top_3_drivers(physician)
    option_b = option_b_high_scores(physician)
    option_c = option_c_deviation_from_5(physician)
    
    print(f"\n{'─'*100}")
    print("REASON CODES - OPTION A: Top 3 Highest Sub-Scores")
    print(f"{'─'*100}")
    for i, reason in enumerate(option_a, 1):
        print(f"  {i}. {reason}")
    
    print(f"\n{'─'*100}")
    print("REASON CODES - OPTION B: All Sub-Scores >= 7 (Outliers)")
    print(f"{'─'*100}")
    for i, reason in enumerate(option_b, 1):
        print(f"  {i}. {reason}")
    
    print(f"\n{'─'*100}")
    print("REASON CODES - OPTION C: Biggest Deviations from Portfolio Mean (5.0)")
    print(f"{'─'*100}")
    for i, reason in enumerate(option_c, 1):
        print(f"  {i}. {reason}")


SAMPLE 1: Internal Medicine-no surgery in FL | NPI: 1003088485.0

Composite Score: 2/10 | Low Risk
  Adequacy:    1.62/10 (40% weight)
  Capacity:    1.00/10 (25% weight)
  Appetite:    3.45/10 (25% weight)
  Environment: 6.24/10 (10% weight)

────────────────────────────────────────────────────────────────────────────────────────────────────
REASON CODES - OPTION A: Top 3 Highest Sub-Scores
────────────────────────────────────────────────────────────────────────────────────────────────────
  1. Violent Crime Rate (Environment 25.0/10)
  2. Practice Size (Appetite 22.0/10)
  3. Pct Uninsured (Environment 17.0/10)

────────────────────────────────────────────────────────────────────────────────────────────────────
REASON CODES - OPTION B: All Sub-Scores >= 7 (Outliers)
────────────────────────────────────────────────────────────────────────────────────────────────────
  1. Violent Crime Rate (Environment 25.0/10)
  2. Practice Size (Appetite 22.0/10)
  3. Pct Uninsured (Environment 17.

## Cell 4: Comparison Table (All Physicians)

In [4]:
print("Generating reason codes for all physicians...\n")

# Add reason code columns to dataframe
df['reasons_option_a'] = df.apply(lambda row: ' | '.join(option_a_top_3_drivers(row)[:3]), axis=1)
df['reasons_option_b'] = df.apply(lambda row: ' | '.join(option_b_high_scores(row)[:3]), axis=1)
df['reasons_option_c'] = df.apply(lambda row: ' | '.join(option_c_deviation_from_5(row)[:3]), axis=1)
df['risk_category'] = df['composite_score'].apply(get_risk_category)

print("✅ Reason codes generated for all physicians\n")

# Show samples by risk category
print("Sample Output by Risk Category:\n")

for category in ['Low Risk', 'Medium Risk', 'High Risk']:
    subset = df[df['risk_category'] == category].head(2)
    
    print(f"\n{category}")
    print("="*120)
    
    for _, row in subset.iterrows():
        print(f"\nNPI: {row['NPI']} | {row['OL_RISK_SPECIALTY_DESC']:30s} | {row['ST']:2s} | Score: {row['composite_score']:.0f}")
        print(f"  Option A (Top 3 scores):        {row['reasons_option_a']}")
        print(f"  Option B (All scores >= 7):    {row['reasons_option_b']}")
        print(f"  Option C (Biggest deviations): {row['reasons_option_c']}")

Generating reason codes for all physicians...

✅ Reason codes generated for all physicians

Sample Output by Risk Category:


Low Risk

NPI: 0.0 | Anesthesiology                 | VA | Score: 3
  Option A (Top 3 scores):        Limits To Premium (Adequacy 13.0/10) | Specialty Risk Tier (Appetite 11.0/10) | State Venue Risk (Appetite 11.0/10)
  Option B (All scores >= 7):    Limits To Premium (Adequacy 13.0/10) | Specialty Risk Tier (Appetite 11.0/10) | State Venue Risk (Appetite 11.0/10)
  Option C (Biggest deviations): Limits To Premium: High (Adequacy 13.0/10) | Specialty Risk Tier: High (Appetite 11.0/10) | State Venue Risk: High (Appetite 11.0/10)

NPI: 0.0 | Dermatology-including minor surgery | FL | Score: 3
  Option A (Top 3 scores):        Population Density (Environment 30.0/10) | Pct Uninsured (Environment 24.0/10) | Violent Crime Rate (Environment 20.0/10)
  Option B (All scores >= 7):    Population Density (Environment 30.0/10) | Pct Uninsured (Environment 24.0/10) | Violen

## Cell 5: Option Comparison Statistics

In [5]:
print("\n" + "="*100)
print("OPTION COMPARISON ANALYSIS")
print("="*100)

# Count how many reasons each option generates per physician
df['count_option_a'] = df['reasons_option_a'].apply(lambda x: len(x.split(' | ')) if x else 0)
df['count_option_b'] = df['reasons_option_b'].apply(lambda x: len(x.split(' | ')) if 'No significant' not in x else 0)
df['count_option_c'] = df['reasons_option_c'].apply(lambda x: len(x.split(' | ')) if x else 0)

print(f"\nOption A (Top 3 Highest Sub-Scores):")
print(f"  Avg reasons per physician: {df['count_option_a'].mean():.2f}")
print(f"  Always generates reasons: {(df['count_option_a'] > 0).sum() / len(df) * 100:.1f}%")

print(f"\nOption B (All Sub-Scores >= 7):")
print(f"  Avg reasons per physician: {df['count_option_b'].mean():.2f}")
print(f"  Physicians with >= 7 score factors: {(df['count_option_b'] > 0).sum() / len(df) * 100:.1f}%")
print(f"  Physicians with NO factors >= 7: {(df['count_option_b'] == 0).sum() / len(df) * 100:.1f}%")

print(f"\nOption C (Biggest Deviations from 5.0):")
print(f"  Avg reasons per physician: {df['count_option_c'].mean():.2f}")
print(f"  Always generates reasons: {(df['count_option_c'] > 0).sum() / len(df) * 100:.1f}%")

print(f"\n" + "="*100)
print("RECOMMENDATION:")
print("="*100)
print(f"""
Option A: Simplest for UW - always shows top 3 drivers. Best for quick scan.
Option B: Risk-focused - only surfaces concerning factors (>=7). Best for exceptions.
Option C: Most nuanced - shows what's unusual for THIS physician. Best for deep review.

Suggestion: Use Option A as default (always populated), with Option B as a filter.
""")


OPTION COMPARISON ANALYSIS

Option A (Top 3 Highest Sub-Scores):
  Avg reasons per physician: 3.00
  Always generates reasons: 100.0%

Option B (All Sub-Scores >= 7):
  Avg reasons per physician: 3.00
  Physicians with >= 7 score factors: 100.0%
  Physicians with NO factors >= 7: 0.0%

Option C (Biggest Deviations from 5.0):
  Avg reasons per physician: 3.00
  Always generates reasons: 100.0%

RECOMMENDATION:

Option A: Simplest for UW - always shows top 3 drivers. Best for quick scan.
Option B: Risk-focused - only surfaces concerning factors (>=7). Best for exceptions.
Option C: Most nuanced - shows what's unusual for THIS physician. Best for deep review.

Suggestion: Use Option A as default (always populated), with Option B as a filter.



## Cell 6: Export Results for Underwriting System

In [6]:
import os
from pathlib import Path

# ============================================================================
# CONFIGURE OUTPUT PARAMETERS
# ============================================================================

# Output directory path
output_directory = r'C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output'

# Output filename (without path)
output_filename = 'physician_uw_tooling_reason_codes.csv'

# ============================================================================

# Combine path and filename
output_path = os.path.join(output_directory, output_filename)

print(f"Output Configuration:")
print(f"  Directory: {output_directory}")
print(f"  Filename:  {output_filename}")
print(f"  Full path: {output_path}")
print()

# Create UW export file with all reason code options
uw_export = df[[
    'NPI',
    'OL_RISK_SPECIALTY_DESC',
    'ST',
    'composite_score',
    'risk_category',
    'adequacy_score',
    'capacity_score',
    'appetite_score',
    'environment_score',
    'reasons_option_a',
    'reasons_option_b',
    'reasons_option_c'
]].copy()

# Save to CSV
try:
    # Create directory if it doesn't exist
    Path(output_directory).mkdir(parents=True, exist_ok=True)
    
    # Write CSV with UTF-8 encoding
    uw_export.to_csv(output_path, index=False, encoding='utf-8')
    
    print(f"✅ SUCCESS!")
    print(f"   Exported {len(uw_export):,} physician records")
    print(f"   File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
    print(f"\n   Saved to: {output_path}")
    print(f"\n   Columns exported:")
    for col in uw_export.columns:
        print(f"     - {col}")
        
except PermissionError:
    print(f"❌ PERMISSION DENIED")
    print(f"   Cannot write to: {output_directory}")
    print(f"   Check that:")
    print(f"     1. Directory exists and you have write access")
    print(f"     2. No file is currently open in Excel")
    print(f"\n   Alternative: Saving to outputs folder...")
    alt_path = f'/mnt/user-data/outputs/{output_filename}'
    uw_export.to_csv(alt_path, index=False, encoding='utf-8')
    print(f"   ✅ Saved to: {alt_path}")
    print(f"❌ ERROR: {e}")
    print(f"   Could not write to: {output_path}")
    print(f"\n   Alternative: Saving to outputs folder...")
    alt_path = f'/mnt/user-data/outputs/{output_filename}'
    uw_export.to_csv(alt_path, index=False, encoding='utf-8')
    print(f"   ✅ Saved to: {alt_path}")

Output Configuration:
  Directory: C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output
  Filename:  physician_uw_tooling_reason_codes.csv
  Full path: C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output\physician_uw_tooling_reason_codes.csv

✅ SUCCESS!
   Exported 165,623 physician records
   File size: 76.57 MB

   Saved to: C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output\physician_uw_tooling_reason_codes.csv

   Columns exported:
     - NPI
     - OL_RISK_SPECIALTY_DESC
     - ST
     - composite_score
     - risk_category
     - adequacy_score
     - capacity_score
     - appetite_score
     - environment_score
     - reasons_option_a
     - reasons_option_b
     - reasons_option_c


## Summary

**Three Reason Code Options Tested:**

- **Option A:** Top 3 highest sub-scores - *Simplest, always has results*
- **Option B:** All scores >= 7 - *Risk-focused, shows outliers only*
- **Option C:** Biggest deviations from 5.0 - *Most nuanced, contextual*

Review the samples above and let me know which option (or combination) works best for your UW team!